In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.vectorstores import FAISS

/var/folders/59/dd24b3q96ss394n23gj0nv840000gn/T/ipykernel_19085/1059762981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [16]:
EMBEDDING_MODEL='text-embedding-3-large'
GPT_MODEL='gpt-5.4'
api_key=os.getenv('OPENAI_API_KEY')

Helpful functions

In [4]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.vectorstores import FAISS

In [5]:
#import essential libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS
from IPython.display import display, Markdown


Excel specific things

In [6]:
#import libraries for document loading
import pandas as pd
from langchain_community.document_loaders import UnstructuredFileLoader, UnstructuredPDFLoader, UnstructuredWordDocumentLoader, UnstructuredPowerPointLoader
from langchain_core.documents import Document


In [27]:
#Prepare the function for retrieval
def retrieval(docs,embedding_model='text-embedding-3-large'):
    #load the embedding model
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    #store in the vector database
    df_faiss = FAISS.from_documents(chunks, embeddings)
    return df_faiss


In [28]:
#Prepare excel Data
def prepare_excel(file_path):
    df=pd.read_excel('Data//Reviews.xlsx')
    #one document /chunk per row
    chunks=[
    Document(page_content='\n'.join(f'{col}:{val}'for col, val in row.items()),
             metadata={"row_index":int(i)}
             )
    for i ,row in df.iterrows()
    ]
    return chunks

  


[Document(metadata={'row_index': 0}, page_content="Before you go any further though, we have a quick favor to ask...\n\nCould you take *60 seconds* to let us know what you think of the course so far (in exchange for this cute cat pic)? :Sure, I'd love to help!\nOur suspicions have been confirmed: *you are *_*awesome*_*!*\n\nStart by selecting the course you're taking below.:Business Analytics Bootcamp (with Python): Zero to Mastery\nHow would you rate this course so far ⭐:5\nWhat do you _*like*_ _*most*_ about the course so far?:nan\nHow can the course be _*improved*_?:nan\n*Anything else* you want to tell us about your experience?:nan")]

In [29]:
#Prepare the function for generation
def generation(df_faiss, query,k=20,gpt_model='gpt-5.4'):
    #Retrieve the content
    retrieved_docs=df_faiss.similarity_search_with_score(query,k=20)
    context="\n".join([doc.page_content for doc,score in retrieved_docs])
    
    #build the prompt for the LLM
    prompt=f"""
    answer the query {query} based on the retrieved context {context} below.
    If the answer is not contained within the context, say "I don't know".
    """
    #call the API
    model=ChatOpenAI(model_name=GPT_MODEL,temperature=0.2,openai_api_key=api_key)
    answer=model.invoke(prompt)
    return display(Markdown(answer.content))


In [17]:
#load the embedding model
embeddings=OpenAIEmbeddings(model=EMBEDDING_MODEL)


In [18]:
df_faiss=FAISS.from_documents(chunks,embeddings)
df_faiss


In [ ]:
#Test the retrieval system
query='Give me most positive reviews'


In [ ]:
import sys
print(sys.version)

3.13.5 (main, Jul 23 2025, 00:18:28) [Clang 20.1.4 ]


In [14]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

[(Document(id='be43fc72-788a-4fec-9f67-3d7b30010138', metadata={'row_index': 27}, page_content="Before you go any further though, we have a quick favor to ask...\n\nCould you take *60 seconds* to let us know what you think of the course so far (in exchange for this cute cat pic)? :Sure, I'd love to help!\nOur suspicions have been confirmed: *you are *_*awesome*_*!*\n\nStart by selecting the course you're taking below.:Statistics with Python: Zero to Mastery\nHow would you rate this course so far ⭐:5\nWhat do you _*like*_ _*most*_ about the course so far?:The instructor is knowlegeable and so far everything has worked.\nHow can the course be _*improved*_?:So far, I'm not tripping over anything that has to be changed.\n*Anything else* you want to tell us about your experience?:Nope. Reviewing Python was good for me."),
  np.float32(1.3709829)),
 (Document(id='b2ff9c08-238c-47c8-a987-b94daec9019f', metadata={'row_index': 5}, page_content="Before you go any further though, we have a quick 

In [20]:
context="\n".join([doc.page_content for doc,score in retrieved_docs])
display(context)

"Before you go any further though, we have a quick favor to ask...\n\nCould you take *60 seconds* to let us know what you think of the course so far (in exchange for this cute cat pic)? :Sure, I'd love to help!\nOur suspicions have been confirmed: *you are *_*awesome*_*!*\n\nStart by selecting the course you're taking below.:Statistics with Python: Zero to Mastery\nHow would you rate this course so far ⭐:5\nWhat do you _*like*_ _*most*_ about the course so far?:The instructor is knowlegeable and so far everything has worked.\nHow can the course be _*improved*_?:So far, I'm not tripping over anything that has to be changed.\n*Anything else* you want to tell us about your experience?:Nope. Reviewing Python was good for me.\nBefore you go any further though, we have a quick favor to ask...\n\nCould you take *60 seconds* to let us know what you think of the course so far (in exchange for this cute cat pic)? :Sure, I'd love to help!\nOur suspicions have been confirmed: *you are *_*awesome*_

In [30]:
#build the prompt for the LLM
prompt=f"""
answer the query {query} based on the retrieved context {context} below.
If the answer is not contained within the context, say "I don't know".
"""

In [31]:
#call the API
model=ChatOpenAI(model_name=GPT_MODEL,temperature=0.2,openai_api_key=api_key)
answer=model.invoke(prompt)
display(Markdown(answer.content))

Here are the most positive reviews from the retrieved context:

1. **Statistics with Python: Zero to Mastery** — **5⭐**
   - “The instructor is knowledgeable and so far everything has worked.”
   - “So far, I'm not tripping over anything that has to be changed.”
   - “Reviewing Python was good for me.”

2. **Time Series Forecasting with Python** — **5⭐**
   - “I love I'm learning the most useful models for time-series, is well explained and comes with the code files!”
   - “It's really easy to follow along the course and you really learn a lot!”
   - “I love it! Thank you a lot!”

There is also a positive review for:

3. **Business Analytics Bootcamp (with Python): Zero to Mastery** — **4⭐**
   - “everyting”
   - “it is already good enough”

The **most positive reviews** are the two **5-star** ones above.

In [32]:
#Test the functions
prepare_excel('Data//Reviews.xlsx')
retrieval(chunks)
generation(df_faiss,'Give me most positive reviews')

Most positive reviews from the context are the 5-star reviews with clearly enthusiastic feedback:

- **Statistics with Python: Zero to Mastery** — “The instructor is knowledgeable and so far everything has worked.”
- **Statistics with Python: Zero to Mastery** — “So engaging and different. Fun exercises and challenges that actually make you think.”
- **Time Series Forecasting with Python** — “I love I'm learning the most useful models for time-series, is well explained and comes with the code files! It's really easy to follow along the course and you really learn a lot!”
- **Statistics with Python: Zero to Mastery** — “A lot of exercises and challenges that help me apply my knowledge and practice. Use of the latest technology such as Google Colab and ChatGpt.”
- **Business Analytics Bootcamp (with Python): Zero to Mastery** — “The content structure” and “I love the flow and pace of the course.”
- **Statistics with Python: Zero to Mastery** — “Simple explanations for beginners.”
- **Python for Business Data Analytics & Intelligence** — “I like the step by step instructions with explanations.”
- **Python for Business Data Analytics & Intelligence** — “good coverage… it is teaching me the same amount in a much faster setting.”

Overall, the most positively reviewed courses in this context appear to be:

1. **Statistics with Python: Zero to Mastery**
2. **Python for Business Data Analytics & Intelligence**
3. **Time Series Forecasting with Python**

In [1]:
import time

def get_current_time()-> str:
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

print(get_current_time())

2026-09-26 23:47:54
